In [ ]:
import time
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp, ndtr, expit
from scipy.linalg import solve, lu_factor, lu_solve
import random
from google.colab import drive
drive.mount('/content/drive')

random.seed(42)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### State Transition Matrix and State Space

Here I want to consider the detailed information on the dynamic question

The state space is given by $s = (x, rc)$ where $x \in \{1, 7\}$ and $rc$ is the replacement cost. Action space $a = \{0, 1\}$ denoting replacing or not replacing. In solving, we denote $G$ RC points, then the total number of states is given by $S = 7G$.

$$
RC_t = \rho_0 + \rho_1 RC_{t-1} + e_t
$$

The transition matrix is given by:

$$
Q = [Q_{gh}]_{g,h = 1}^G, \quad Q_{gh} = \Pr(RC_{t + 1} \in \text{ grid } h \mid RC_{t} = r^g)
$$

Denote grid space as $\Delta$ then the interior grid cells is:

$$
Q_{gh} = \Phi\left( \frac{r^h + \Delta / 2 - (\rho_0 + \rho_1 r^g)}{\sigma_\rho} \right) - \Phi\left( \frac{r^h - \Delta / 2 - (\rho_0 + \rho_1 r^g)}{\sigma_\rho} \right)
$$

with the corresponding tail formulas for the first and last bins.

In terms of Mileage transition, we can consider the transition is given by:

$$
x^\prime = \begin{cases}
1 & \text{ if } a = 1 \\
\min(7, x + 1) & \text{ if } a = 0
\end{cases}
$$

We denote $M_0(x, x^\prime) = \mathbf{1}\{x^\prime = x^\prime = \min(7, x + 1)\}$ and $M_1(x, x^\prime) = \mathbf{1}\{x^\prime = 1\}$. Then the full state transition is given by:

$$
F_0 = M_0 \otimes Q, \quad F_1 M_1 \otimes Q
$$

Therefore, for current $s = (x, rc)$ and the next state, we have:

$$
F_a(s, s^\prime) = \Pr(s^\prime \mid s, a)
$$

### Bellman Equation

Let $V(s)$ be the ex ante function. We can write the action specific continuation value as:

$$
EV_a(s) = \sum_{s^\prime = 1}^S F_a(s, s^\prime) V(s^\prime)
$$

Then the Bellman Equation is given by:

$$
V(s) = \mathbb{E}_\varepsilon [\max_{a \in \{0, 1\}} (\bar{u}(s, a) + \varepsilon_a + \beta EV_a(s))]
$$

where the stock is a Type-1 extreme value. Then we can write as:

$$
V(s) = \log(\exp(\bar{u}(s, 0) + \beta EV_0(s)) + \exp(\bar{u}(s, 1) + \beta EV_1(s))) + \gamma
$$

(Side Note: Here I leverage the term from the fact of Appendix A)

where $\gamma$ is a Euler Constant. Then we can substitute the utility term and we have:

$$
V(s) = \log(\exp\{-\theta_1 x_s + \beta EV_0(s)\} + \exp\{-\theta_2 rc_s + \beta EV_1(s)\}) + \gamma
$$

## CCP and NXFP

In the logit CCP implied by Bellman Equation, we can write

$$
\Pr(a = 1 \mid s) = \frac{\exp(-\theta rc_s + \beta EV_1(s))}{ \exp\{-\theta_1 x_s + \beta EV_0(s)\} + \exp\{-\theta_2 rc_s + \beta EV_1(s)\} }
$$

and

$$
\Pr(a = 0 \mid s) = \frac{ \exp\{-\theta_1 x_s + \beta EV_0(s)\} }{ \exp\{-\theta_1 x_s + \beta EV_0(s)\} + \exp\{-\theta_2 rc_s + \beta EV_1(s)\} }
$$

Then we can write the State-level likelihood, which aggregates the likelihood to the state level. Once the state $s$ is fixed, we can write

$$
N_1(s) = \#\{i: a_i = 1 \text{ in state }s\}, \quad
N_0(s) = \#\{i: a_i = 0 \text{ in state }s\}
$$

Then the log-likelihood can be written as:

$$
\ell(\theta) = \sum_{s=1}^S [N_1(s)\log \Pr(a = 1 \mid s; \theta) + N_0(s)\log \Pr(a = 0 \mid s; \theta)]
$$



We can write the nested fixed point algorithm using the below condition:

We can denote

$$
EV = \begin{pmatrix}
EV_1 \\
EV_2
\end{pmatrix}
$$

and define the Bellman operator $T_\theta$. Then the fixed point is given by:

$$
EV = T_\theta(EV)
$$

and the numerical expression ig: $EV^{k} = T_\theta(EV^{k-1})$. By the Banach's fixed point theorem, we can have: $EV^{k} \to EV_\theta$.

### Hotz-Miller Inversion

Based on the Bellman Equation and we can write it using the CCP condition as:

$$
V(s) = \sum_{a \in \{0, 1\}} P(a \mid s) [\bar{u}(a, s) + \beta \sum_{s^\prime} F_a(s, s^\prime) V(s^\prime)] + \sum_{a \in \{0, 1\}} P(a \mid s) \mathbb{E}[\varepsilon_a \mid a \text{ chosen } s].
$$

Under the Type-1 extreme value assumption we have:

$$
\mathbb{E}[\varepsilon_a \mid a \text{ chosen }, s] = \gamma - \log P(a \mid s)
$$

Therefore,

$$
V(s) = \sum_{a \in \{0, 1\}} P(a \mid s) \left[ \bar{u}(s, a) + \beta \sum_{s^\prime} F_a(s, s^\prime) V(s^\prime) + \gamma - \log P(a \mid s) \right]
$$

Now write in the vector form and we can write

$$
P_a = \begin{pmatrix}
P(a \mid 1) \\
\cdots \\
P(a \mid S)
\end{pmatrix} \quad
\bar{u}_a = \begin{pmatrix}
\bar{u}(1, a) \\
\cdots \\
\bar{u}(S, a)
\end{pmatrix}
$$

We can write the diagonal matrix whose diagonal entries are given $P_a$. Then the Bellman equation becomes

$$
V = \sum_{a \in \{0, 1\}} D(P_a)(\bar{u}_a + \gamma \mathbb{1} - \log P_a) + \beta \sum_a D(P_a) F_aV.
$$

Move to the LHS, inverse it and we have:

$$
V = \left[ I - \beta \sum_a D(P_a) F_a\right]^{-1} \sum_a D(P_a)(\bar{u}_a + \gamma \mathbb{1} - \log P_a)
$$

## Forward Simulation Method in Dynamic Estimation

### (C. 1) Method: Forward simulation using estimated CCPs

Fix an initial state $s$ and for simulation replication $r = 1, \ldots, R$, generate a path $\{s_\tau^{(r)}, a_\tau^{(r)}\}_{\tau = 0}^{T_{sim}}$ as follows:

First set $s_0^{(r)} = s$ and at each date $\tau$:

1. Draw the action from the estimated CCP:

$$
a_\tau^{(r)} \sim \text{Bernoulli}(\hat{P}(1 \mid s_\tau^{(r)})).
$$

2. Given $a_\tau^{(r)}$, update mileage deterministically:

$$
x_{\tau + 1}^{(r)} = \begin{cases}
1, & a_\tau^{(r)} = 1 \\
\min(7, x_\tau^{(r)} + 1), & a_\tau^{(r)} = 0.
\end{cases}
$$

3. Update replacement cost using the AR(1) process as the parameters obtained in the reduced form part.

$$
RC_{\tau + 1}^{(r)} = \hat{\rho}_0 + \hat{\rho}_1 RC_{\tau}^{(r)} + \hat{\sigma}_\rho \eta_{\sigma + 1}^{(r)}, \quad \eta_{\tau + 1}^{(r)} \sim N(0, 1)
$$

Since $RC$ is discrete, we can use the same method that we discussed in terms of the projection into the corresponding grids.

For the simulated path, define the truncation return:

$$
G^{(1, r)}(s; \theta) = \sum_{\tau = 0}^{T_{sim}} \beta^{\tau} [\bar{u}(s_\tau^{(r)}, a_\tau^{(r)}; \theta) + \gamma - \log \hat{P}(a_\tau^{(r)} \mid s_\tau^{(r)} ].
$$

where $\gamma$ is again the Euler's constant. The term $\gamma - \log \hat{P}(a \mid s)$ is the conditional expectation of the selected Type-I extreme value shock under the CCP representation:

$$
\mathbb{E}_a[\varepsilon_a \mid a \text{ chosen }, s] = \gamma - \log P(a \mid s)
$$

Therefore, the simulated return in this expression is exactly the finite-horizon approximation to the value of following policy $\hat{P}$. This is the quantity prescribed by Eq. (5) in the assignment. Average across simulation replications to obtain the value estimator

$$
\hat{V}^{(1)}(s; \theta) = \frac{1}{R} \sum_{r=1}^R G^{(1, r)}(s; \theta)
$$

As $R \to \infty$, and $T_{sim} \to \infty$, this converges to the policy value $V^{\hat{P}}(s; \theta)$, provided the CCPs are bounded away from $0$ and $1$ and the transition process is correctly approximated.

Once $\hat{V}^{(1)}(s; \theta)$ has been computed for all states $s$ the choice specific continuation values are obtained in the same way as:

$$
\hat{EV}_a^{(1)}(s; \theta) = \sum_{s^\prime} \hat{F}_a(s, s^\prime) \hat{V}^{(1)}(s^\prime, \theta)
$$

Then the implied choice probabilities used in the likelihood are

$$
\hat{p}^{(1)}(s; \theta) = \frac{ \exp(-\theta_2 rc_s + \beta \hat{EV}_1^{(1)}(s; \theta))}{ \exp(-\theta_1 x_s + \beta \hat{EV}_0^{(1)}(s; \theta)) + \exp(-\theta_2 rc_s + \beta \hat{EV}_1^{(1)}(s; \theta) ) }
$$

and $\hat{p}_0^1(s; \theta) = 1 - \hat{p}_1^{(1)}(s; \theta)$. The corresponding state-level simulated log-likelihood is similarly constructed as:

$$
\ell^{(1)}(\theta) = \sum_{s=1}^S [N_1(s) \log \hat{p}_1^{(1)}(s; \theta) + N_0(s) \log \hat{p}_0^{(1)}(s; \theta)]
$$

So the estimator is

$$
\hat{\theta}^{(1)} = \arg\max_\theta \ell^{(1)}(\theta)
$$

This estimator is a two-step simulated pseudo-likelihood estimator: conditional on first-stage estimates $\hat{P}$ and $\hat{F}$, the continuation value is approximated by simulation rather than by fixed-point iteration.

### (C. 2) Method: Forward simulation by drawing utility shocks directly

The second simulation method replaces the analytical correction term $\gamma - \log \hat{P}(a \mid s)$ by explicit simulation of the Type-I extreme value shocks. Again fix an initial state $s$, and for each simulation replication $r = 1, \ldots, R$, generate a path $\{s_\tau^{(r)}, a_\tau^{(r)}\}_{\tau = 0}^{T_{sim}}$. Set

$$
s_0^{(r)} = s.
$$

At each draw $\tau$, draw $\varepsilon_{0\tau}^{(r)}, \varepsilon_{1\tau}^{(r)}$ independently from the Type-I extreme value distribution. Then choose action according to the rule

$$
a_{\tau}^{(r)} = 1 \iff \log \frac{ \hat{P}(1 \mid s_\tau^{(r)})}{\hat{P}(0 \mid s_\tau^{(r)})} + \varepsilon_{1\tau}^{(r)} - \varepsilon_{0\tau}^{(r)} > 0.
$$

This choice rule is valid because the difference of two independent Type-I extreme value shocks has the logistic distribution. Hence

$$
\Pr(a = 1 \mid s) = \Pr(s_1 - s_0 > -\log \frac{ \hat{P}(1 \mid s_\tau^{(r)})}{\hat{P}(0 \mid s_\tau^{(r)})} = \hat{P}(1 \mid s)
$$

Therefore this procedure exactly reproduces the estimated CCP at each state. This is the content of Eq. (6) in the assignment.

Given the chosen action, update mileage and replacement cost in exactly the same way as in part (c.1). For the simulated path, we can correspondingly define the result as:

$$
\hat{V}^{(2)}(s; \theta) = \frac{1}{R} \sum_{r=1}^R G^{(2, r)}(s; \theta).
$$

where the truncated discounted return

$$
G^{(2, r)}(s; \theta) = \sum_{\tau = 0}^{T_{sim}} \beta^{\tau} [\bar{u}(s_\tau^{(r)}, a_\tau^{(r)}; \theta) + \varepsilon_{a_{\tau}^{(r)}, \tau}^{(r)} ].
$$

Once $\hat{V}^{(2)}(s; \theta)$ is computed for all states, we can correspondingly define:

$$
\hat{EV}_a^{(2)}(s; \theta) = \sum_{s^\prime} \hat{F}_a(s, s^\prime) \hat{V}^{(2)}(s^\prime, \theta)
$$

Then the implied choice probabilities used in the likelihood are

$$
\hat{p}^{(2)}(s; \theta) = \frac{ \exp(-\theta_2 rc_s + \beta \hat{EV}_1^{(2)}(s; \theta))}{ \exp(-\theta_1 x_s + \beta \hat{EV}_0^{(2)}(s; \theta)) + \exp(-\theta_2 rc_s + \beta \hat{EV}_1^{(2)}(s; \theta) ) }
$$

and $\hat{p}_0^2(s; \theta) = 1 - \hat{p}_1^{(2)}(s; \theta)$. The corresponding state-level simulated log-likelihood is similarly constructed as:

$$
\ell^{(2)}(\theta) = \sum_{s=1}^S [N_1(s) \log \hat{p}_1^{(2)}(s; \theta) + N_0(s) \log \hat{p}_0^{(2)}(s; \theta)]
$$

So the estimator is

$$
\hat{\theta}^{(2)} = \arg\max_\theta \ell^{(2)}(\theta)
$$

### Comparision between (C. 1) and (C. 2)$ methods:

The two simulation procedures approximate the same policy value $V^{\hat{P}}(s; \theta)$. The selected shock is integrated out analytically in the (C. 1)$, so the simulated path uses only the deterministic utility component plus this correction term. In part (C. 2), the shocks are drawn explicitly and the action is selected using the implied threshold rule. Thus part (C.1) generally has lower simulation variance, while part (C.2) is closer to the original random-utility representation of the model. Both are valid implementations of the forward-simulation estimator described in part (C).

## Regression Based Estimation without the Dynamic Setup

To make the notation simplier, we denote, for each date $t$,

$$
p_{1, t} = \Pr(a_t = 1 \mid x_t = x, RC_t), \quad p_{0, t}(x) = 1 - p_{1, t}(x)
$$

Since $RC_t$ is constant within month $t$, there are one dimensional CCPs in $x$ for each $t$. Let $m(x) = \min(x + 1, 7)$.

Start from the logit choice-probability identity. Under the tyle -I extreme value shocks, we have:

$$
\log \frac{p_{0, t}(x)}{p_{1, t}(x)} = [-\theta_1 x + \beta \mathbb E_t V_{t + 1}(m(x), RC_{t + 1})] - [-\theta_2 RC_t + \beta \mathbb E_t[V_{t + 1}(1, RC_{t + 1})].
$$

Rearrange and we have:

$$
\log \frac{p_{0, t}(x)}{p_{1, t}(x)} = -\theta_1 x + \theta_2 RC_t + \beta \mathbb E_t  [V_{t + 1}(m(x), RC_{t + 1}) - V_{t + 1}(1, RC_{t + 1})].
$$

ow use the logit identity for the next period. For any next-period state $(x^\prime, RC_{t + 1})$:

$$
V_{t + 1}(x^\prime, RC_{t + 1}) = v_{1, t + 1}(x^\prime, RC_{t + 1}) - \log p_{1, t + 1}(x^\prime) + \gamma
$$

where $v_{1, t + 1}(x^\prime, RC_{t + 1}) = -\theta_2 RC_{t + 1} + \beta \mathbb E_t[ V_{t + 2}(1, RC_{t + 2}) \mid RC_{t + 1}]$. Crucially, $v_{1, t + 1}(x^\prime, RC_{t + 1})$ does not depend on $x^\prime$ because if action $1$ is chosen at time $t + 1$ then mileage is always reset to $1$ and the future law of $RC$ is exogenous and independent on the current mileage. Therefore, when we difference two next-period states with the same $RC_{t + 1}$ but different mileage, the common terms cancel:

$$
V_{t + 1}(m(x), RC_{t + 1}) - V_{t + 1}(1, RC_{t + 1}) = \log \frac{p_{1, t + 1}(1)}{p_{1, t + 1}(m(x))}
$$

Substitute the above tow equations. This gives the exact Scott-style linear relation:

$$
\log \frac{p_{0, t}(x)}{p_{1, t}(x)} + \beta \mathbb E_t [\log \frac{p_{0, t}(m(x))}{p_{1, t}(1)}] = -\theta_1 x + \theta_2 RC + \varepsilon_1 + \varepsilon_2
$$

To derive the estimator we can write:

$\hat{\Lambda}_t(x) = \log \frac{1 - \hat{p}_{1, t}(x)}{\hat{p}_{1, t}(x)}$ and because the dynamic term requires $t + 1$, define the realized continuation proxy $\hat{C}_t(x) = \log \frac{\hat{p}_{1, t + 1}(m(x))}{\hat{p}_{1, t + 1}(1)}$. Then the regression dependent variable is

$$
\hat{Y}_t(x) = \hat{\Lambda}_t(x) + \hat{C}_t(x) + u_t(x)
$$

where the residual $u_t(x)$ contains first-stage estimation error and the difference between the realized next-period continuation term and its conditional expectation. Under rational expectations and exogeneity of $RC_t$,

$$
\mathbb E_t[u_t(x) \mid x, RC_t] = 0
$$

So the estimator can be written as weighted least squares over $(t, x_t)$ with $w_t = n_t(x)$ in the setup:

$$
\hat{\theta}^{reg} = \arg\min_{\theta_1, \theta_2} \sum_{t=1}^{T - 1} \sum_{x=1}^7 w_t(x) [\hat{Y}_t(x) + \theta_1 x - \theta_2 RC_t]^2
$$

In [ ]:

# basic setup

DATA_PATH = '/content/drive/MyDrive/ECON662HWs/ddc_pset.csv'
# DATA_PATH = 'ddc_pset.csv'

beta = 0.95
K = 8
ccp_clip = 1e-10                    # numerical clip so probabilities never hit exactly 0 or 1

# First-stage CCP smoothing controls
# alpha is a small pseudo-count; Sx and Sr provide local averaging.
alpha_ccp = 1.0
w_center = 0.50
w_neighbor = 0.25

bellman_tol = 1e-10
bellman_tol_stage1 = 1e-6
bellman_tol_stage2 = 1e-11
bellman_maxiter = 5000
gamma_euler = 0.5772156649015329

theta0 = np.array([0.30, 0.35], dtype=np.float64)   # initial guess for (theta1, theta2)

# Load data and organize panel

df = pd.read_csv(DATA_PATH).sort_values(["i", "t"]).reset_index(drop=True)
print(df.columns)

x_panel = df.pivot(index="i", columns="t", values="x").to_numpy(dtype=np.int16)
a_panel = df.pivot(index="i", columns="t", values="a").to_numpy(dtype=np.int8)

rc_path = (
    df[["t", "rc"]]
    .drop_duplicates(subset="t")
    .sort_values("t")["rc"]
    .to_numpy(dtype=np.float64)
)

N, T = x_panel.shape
x_flat = x_panel.ravel(order="C").astype(np.int16) - 1
a_flat = a_panel.ravel(order="C").astype(np.int8)

# First stage for RC_t = rho0 + rho1 * RC_{t-1} + e_t

Y = rc_path[1:]
X = np.column_stack([np.ones(T - 1), rc_path[:-1]])
# Run OLS using numpy least squares.
rho_ols = np.linalg.lstsq(X, Y, rcond=None)[0]

# Pull out the intercept estimate.
rho0_hat = float(rho_ols[0])
# Pull out the persistence estimate and clip it slightly away from |rho1| >= 1
# to avoid numerical issues in subsequent transition calculations.
rho1_hat = float(np.clip(rho_ols[1], -0.999, 0.999))


# Compute OLS residuals e_t.
resid = Y - X @ rho_ols
# Estimate sigma_rho as the residual standard deviation.
sigma_rho_hat = float(np.sqrt(np.mean(resid ** 2)))
sigma_rho_hat = max(sigma_rho_hat, 1e-12)

print("First-stage estimates for RC_t:")
print(f"rho0_hat      = {rho0_hat:.6f}")
print(f"rho1_hat      = {rho1_hat:.6f}")
print(f"sigma_rho_hat = {sigma_rho_hat:.6f}")

# Discretize RC and aggregate state counts

rc_min = float(rc_path.min())
rc_max = float(rc_path.max())

# Build K+1 equally spaced bin edges covering the observed RC support.
edges = np.linspace(rc_min, rc_max, K + 1)
# Use the midpoint of each bin as the representative RC grid value.
r_grid = 0.5 * (edges[:-1] + edges[1:])


# For each month t, assign observed RC_t to one of the K bins.
# searchsorted(..., side="right") - 1 implements interval assignment.
rc_bin_by_month = np.searchsorted(edges, rc_path, side="right") - 1
# Clip to make sure the assigned bin index stays in {0,...,K-1}.
rc_bin_by_month = np.clip(rc_bin_by_month, 0, K - 1).astype(np.int16)

# Repeat the month-level RC bin sequence once for each bus.
# Because x_flat and a_flat were created in bus-major row order,
# this tiled vector lines up exactly with those flattened panels.
rc_bin_flat = np.tile(rc_bin_by_month, N)

S = 7 * K
# Encode each state (x, g) as a single integer state id.
# Since x_flat is 0,...,6 and g is 0,...,K-1, the formula x*K + g is one-to-one.
state_id = x_flat * K + rc_bin_flat

# Count how many times action a=1 is observed in each state.
N1_vec = np.bincount(state_id[a_flat == 1], minlength=S).astype(np.float64)
# Count how many times action a=0 is observed in each state.
N0_vec = np.bincount(state_id[a_flat == 0], minlength=S).astype(np.float64)
N_vec = N1_vec + N0_vec
# Boolean mask for states that are actually observed in the data.
occupied = N_vec > 0

N1_mat = N1_vec.reshape(7, K)
N0_mat = N0_vec.reshape(7, K)
N_mat = N_vec.reshape(7, K)

# Empirical CCP = observed replacement frequency in each state.
empirical_ccp = np.divide(
    N1_vec,
    N_vec,
    out=np.zeros_like(N1_vec),
    where=(N_vec > 0)
)

# Print how many of the S states are actually visited.
print(f"\nOccupied states = {int(occupied.sum())} out of {S}")


# I tried this to use the $K = 50, 100$ and I find that if I don't smooth this
# the result will bias the whole estimation because there are a lot of states
# to be empty.

# Build a one-dimensional local averaging matrix of size n x n.
# Each state puts weight w_center on itself and w_neighbor on each immediate neighbor.
# Rows are normalized to sum to 1.
def local_smoother_matrix(n, w_center=0.50, w_neighbor=0.25):
    M = np.zeros((n, n), dtype=np.float64)
    idx = np.arange(n)
    M[idx, idx] = w_center # own-state weight
    M[idx[:-1], idx[1:]] = w_neighbor # right neighbor weight
    M[idx[1:], idx[:-1]] = w_neighbor # left neighbor weight
    M /= M.sum(axis=1, keepdims=True) # row normalization
    return M

# Smoother along the mileage dimension.
Sx = local_smoother_matrix(7, w_center=w_center, w_neighbor=w_neighbor)
Sr = local_smoother_matrix(K, w_center=w_center, w_neighbor=w_neighbor)

# Smooth replacement counts over nearby mileage and nearby RC states.
N1_smooth = Sx @ N1_mat @ Sr.T
N_smooth = Sx @ N_mat @ Sr.T

# Convert smoothed counts into probabilities using pseudo-counts.
ccp_smooth_mat = (N1_smooth + alpha_ccp) / (N_smooth + 2.0 * alpha_ccp)

# Clip probabilities away from 0 and 1 for numerical stability.
ccp_smooth_mat = np.clip(ccp_smooth_mat, ccp_clip, 1.0 - ccp_clip)
ccp_smooth = ccp_smooth_mat.reshape(-1)


# RC transition matrix Q_rc

# Given current grid point r_grid[g], the AR(1) implies next-period RC is normal with mean:
#   rho0_hat + rho1_hat * r_grid[g]
mu_rc = rho0_hat + rho1_hat * r_grid[:, None]

# Standardized upper bin edges for all current-state / next-bin combinations.
upper = (edges[1:][None, :] - mu_rc) / sigma_rho_hat
# Standardized lower bin edges for all current-state / next-bin combinations.
lower = (edges[:-1][None, :] - mu_rc) / sigma_rho_hat
Q_rc = ndtr(upper) - ndtr(lower)
Q_rc /= Q_rc.sum(axis=1, keepdims=True)

# Matrix-free transition operators
x_next_keep = np.array([1, 2, 3, 4, 5, 6, 6], dtype=np.int64)
x_reset = 0


# deterministic utility bases on the state grid
x_state_mat = np.repeat(np.arange(1, 8), K).reshape(7, K).astype(np.float64)
rc_state_mat = np.tile(r_grid, 7).reshape(7, K).astype(np.float64)

u0_base = -x_state_mat                 # maintenance cost base
u1_base = -rc_state_mat                # replacement cost base

u0_base_vec = u0_base.reshape(-1)
u1_base_vec = u1_base.reshape(-1)

# Build dense F0 and F1 once.
# Current state ordering is row-major on a 7 x K grid: state = x*K + g
F0 = np.zeros((S, S), dtype=np.float64)
F1 = np.zeros((S, S), dtype=np.float64)

for x in range(7):
    row_slice = slice(x * K, (x + 1) * K)

    keep_x = int(x_next_keep[x])
    col_slice_keep = slice(keep_x * K, (keep_x + 1) * K)
    F0[row_slice, col_slice_keep] = Q_rc

    col_slice_reset = slice(x_reset * K, (x_reset + 1) * K)
    F1[row_slice, col_slice_reset] = Q_rc

Fdiff = F1 - F0
I_S = np.eye(S, dtype=np.float64)

# two-action log-sum-exp and CCP map

def lse2(a, b):
    m = np.maximum(a, b)
    return m + np.log(np.exp(a - m) + np.exp(b - m))

def choice_prob_from_V_flat(theta1, theta2, V_flat):
    EV0 = F0 @ V_flat
    EV1 = F1 @ V_flat

    W0 = theta1 * u0_base_vec + beta * EV0
    W1 = theta2 * u1_base_vec + beta * EV1

    delta = W1 - W0
    p1 = expit(delta)

    return np.clip(p1, ccp_clip, 1.0 - ccp_clip), W0, W1, EV0, EV1

# NFXP Bellman solver with warm start

_nfxp_cache = {
    "eta": None,
    "V": np.zeros(S, dtype=np.float64),
}

def bellman_operator_nfxp(theta1, theta2, V_flat):
    EV0 = F0 @ V_flat
    EV1 = F1 @ V_flat

    W0 = theta1 * u0_base_vec + beta * EV0
    W1 = theta2 * u1_base_vec + beta * EV1

    return gamma_euler + lse2(W0, W1)

def solve_value_nfxp(theta1, theta2, tol=bellman_tol, maxiter=bellman_maxiter):
    V = _nfxp_cache["V"].copy()

    for it in range(maxiter):
        V_new = bellman_operator_nfxp(theta1, theta2, V)
        err = np.max(np.abs(V_new - V))
        V = V_new
        if err < tol:
            _nfxp_cache["V"] = V.copy()
            return V, it + 1

    _nfxp_cache["V"] = V.copy()
    return V, maxiter

# NFXP log-likelihood and exact gradient in eta-space
#    eta = log(theta), so theta = exp(eta) enforces positivity

def nfxp_objective_and_grad_eta(eta, inner_tol):
    theta = np.exp(eta)
    theta1 = float(theta[0])
    theta2 = float(theta[1])

    V, n_inner = solve_value_nfxp(theta1, theta2, tol=inner_tol)

    p1, W0, W1, EV0, EV1 = choice_prob_from_V_flat(theta1, theta2, V)
    ll = np.sum(N1_vec * np.log(p1) + N0_vec * np.log1p(-p1))

    p0 = 1.0 - p1

    # Bellman Jacobian wrt V:
    # D_V T = beta * [diag(p0) F0 + diag(p1) F1]
    DVT = beta * (p0[:, None] * F0 + p1[:, None] * F1)
    A = I_S - DVT

    # dT/dtheta_j
    dT_dtheta1 = p0 * u0_base_vec
    dT_dtheta2 = p1 * u1_base_vec

    # Implicit differentiation: (I - D_V T) dV/dtheta = dT/dtheta
    dV_dtheta1 = solve(A, dT_dtheta1, assume_a='gen', check_finite=False)
    dV_dtheta2 = solve(A, dT_dtheta2, assume_a='gen', check_finite=False)

    # delta = W1 - W0
    dDelta_dtheta1 = -u0_base_vec + beta * (Fdiff @ dV_dtheta1)
    dDelta_dtheta2 =  u1_base_vec + beta * (Fdiff @ dV_dtheta2)

    # For logit grouped likelihood:
    # d ell / d delta = N1 - N * p1
    score_weight = N1_vec - N_vec * p1
    score_theta1 = np.sum(score_weight * dDelta_dtheta1)
    score_theta2 = np.sum(score_weight * dDelta_dtheta2)

    # Chain rule: d ell / d eta_j = d ell / d theta_j * theta_j
    grad_eta = -np.array([
        score_theta1 * theta1,
        score_theta2 * theta2
    ], dtype=np.float64)

    return -ll, grad_eta, p1, V, n_inner

def nfxp_fun_stage1(eta):
    f, g, _, _, _ = nfxp_objective_and_grad_eta(eta, inner_tol=bellman_tol_stage1)
    return f, g

def nfxp_fun_stage2(eta):
    f, g, _, _, _ = nfxp_objective_and_grad_eta(eta, inner_tol=bellman_tol_stage2)
    return f, g

# HM direct solve and exact gradient in eta-space

p1_first_stage_vec = ccp_smooth_mat.reshape(-1)
p0_first_stage_vec = 1.0 - p1_first_stage_vec

neglog_p0_vec = -np.log(p0_first_stage_vec)
neglog_p1_vec = -np.log(p1_first_stage_vec)

# HM operator: V = b(theta) + beta * P_sigma V
P_sigma = p0_first_stage_vec[:, None] * F0 + p1_first_stage_vec[:, None] * F1
A_hm = I_S - beta * P_sigma
lu_hm, piv_hm = lu_factor(A_hm, check_finite=False)

# These derivatives are constant in theta because A_hm is fixed
dVhm_dtheta1_const = lu_solve((lu_hm, piv_hm), p0_first_stage_vec * u0_base_vec, check_finite=False)
dVhm_dtheta2_const = lu_solve((lu_hm, piv_hm), p1_first_stage_vec * u1_base_vec, check_finite=False)

def solve_value_hm(theta1, theta2):
    u0 = theta1 * u0_base_vec
    u1 = theta2 * u1_base_vec

    b = (
        p0_first_stage_vec * (u0 + gamma_euler + neglog_p0_vec)
        + p1_first_stage_vec * (u1 + gamma_euler + neglog_p1_vec)
    )
    V = lu_solve((lu_hm, piv_hm), b, check_finite=False)
    return V

def hm_objective_and_grad_eta(eta):
    theta = np.exp(eta)
    theta1 = float(theta[0])
    theta2 = float(theta[1])

    V = solve_value_hm(theta1, theta2)

    p1, W0, W1, EV0, EV1 = choice_prob_from_V_flat(theta1, theta2, V)
    ll = np.sum(N1_vec * np.log(p1) + N0_vec * np.log1p(-p1))

    dDelta_dtheta1 = -u0_base_vec + beta * (Fdiff @ dVhm_dtheta1_const)
    dDelta_dtheta2 =  u1_base_vec + beta * (Fdiff @ dVhm_dtheta2_const)

    score_weight = N1_vec - N_vec * p1
    score_theta1 = np.sum(score_weight * dDelta_dtheta1)
    score_theta2 = np.sum(score_weight * dDelta_dtheta2)

    grad_eta = -np.array([
        score_theta1 * theta1,
        score_theta2 * theta2
    ], dtype=np.float64)

    return -ll, grad_eta, p1, V

def hm_fun(eta):
    f, g, _, _ = hm_objective_and_grad_eta(eta)
    return f, g

# Run estimators

eta0 = np.log(theta0)

# -------- NFXP: two-stage BFGS --------
t0 = time.perf_counter()

res_nfxp_stage1 = minimize(
    nfxp_fun_stage1,
    eta0,
    method="BFGS",
    jac=True,
    options={
        "gtol": 1e-4,
        "maxiter": 150,
        "disp": False,
    },
)

res_nfxp = minimize(
    nfxp_fun_stage2,
    res_nfxp_stage1.x,
    method="BFGS",
    jac=True,
    options={
        "gtol": 1e-4,
        "maxiter": 200,
        "disp": False,
    },
)

time_nfxp = time.perf_counter() - t0

theta_nfxp = np.exp(res_nfxp.x)
ll_nfxp, grad_nfxp, p1_nfxp, V_nfxp, nfxp_last_inner = nfxp_objective_and_grad_eta(
    res_nfxp.x, inner_tol=bellman_tol_stage2
)
ll_nfxp = -ll_nfxp

# -------- HM: one-stage BFGS with exact gradient --------
t0 = time.perf_counter()

res_hm = minimize(
    hm_fun,
    eta0,
    method="BFGS",
    jac=True,
    options={
        "gtol": 1e-8,
        "maxiter": 200,
        "disp": False,
    },
)

time_hm = time.perf_counter() - t0

theta_hm = np.exp(res_hm.x)
obj_hm, grad_hm, p1_hm, V_hm = hm_objective_and_grad_eta(res_hm.x)
ll_hm = -obj_hm

# Matrix-size report

dense_transition_shape = (S, S)
dense_transition_elements = S * S
dense_transition_memory_mb = dense_transition_elements * 8 / (1024 ** 2)

qrc_shape = Q_rc.shape
qrc_elements = Q_rc.size
qrc_memory_mb = qrc_elements * 8 / (1024 ** 2)

matrix_report = pd.DataFrame({
    "object": [
        "State space",
        "Dense F0/F1",
        "Stored RC transition Q_rc",
        "Smoothed CCP grid",
    ],
    "shape": [
        f"{S}",
        f"{dense_transition_shape}",
        f"{qrc_shape}",
        f"{ccp_smooth_mat.shape}",
    ],
    "elements": [
        S,
        dense_transition_elements,
        qrc_elements,
        ccp_smooth_mat.size,
    ],
    "approx_memory_MB": [
        np.nan,
        dense_transition_memory_mb,
        qrc_memory_mb,
        ccp_smooth_mat.size * 8 / (1024 ** 2),
    ],
})

# Final report

print("\n================ NFXP results ================")
print(f"success                = {res_nfxp.success}")
print(f"message                = {res_nfxp.message}")
print(f"loglike                = {ll_nfxp:.6f}")
print(f"theta1_hat             = {theta_nfxp[0]:.6f}")
print(f"theta2_hat             = {theta_nfxp[1]:.6f}")
print(f"runtime_sec            = {time_nfxp:.4f}")
print(f"last_inner_iterations  = {nfxp_last_inner}")

print("\n============= Hotz-Miller results =============")
print(f"success                = {res_hm.success}")
print(f"message                = {res_hm.message}")
print(f"loglike                = {ll_hm:.6f}")
print(f"theta1_hat             = {theta_hm[0]:.6f}")
print(f"theta2_hat             = {theta_hm[1]:.6f}")
print(f"runtime_sec            = {time_hm:.4f}")
print(f"HM value solve         = direct linear solve")

summary = pd.DataFrame({
    "method": ["NFXP", "Hotz-Miller MLE"],
    "theta1_hat": [theta_nfxp[0], theta_hm[0]],
    "theta2_hat": [theta_nfxp[1], theta_hm[1]],
    "rho0_hat": [rho0_hat, rho0_hat],
    "rho1_hat": [rho1_hat, rho1_hat],
    "sigma_rho_hat": [sigma_rho_hat, sigma_rho_hat],
    "loglike": [ll_nfxp, ll_hm],
    "runtime_sec": [time_nfxp, time_hm],
    "state_space_S": [S, S],
})

print("\n================ Summary table ================")
print(summary.to_string(index=False))

print("\n================ Matrix-size report ================")
print(matrix_report.to_string(index=False))

print("\nFirst 10 occupied-state empirical CCPs:")
print(empirical_ccp[occupied][:10])

print("\nFirst 10 occupied-state smoothed CCPs:")
print(ccp_smooth[occupied][:10])

print("\nFirst 10 occupied-state model CCPs from NFXP:")
print(p1_nfxp[occupied][:10])

print("\nFirst 10 occupied-state model CCPs from HM:")
print(p1_hm[occupied][:10])

print("\nRC grid midpoints:")
print(r_grid)


## Now I want to consider the dynamic estimation of the part (c) and (d)



Index(['i', 't', 'a', 'x', 'rc'], dtype='object')
First-stage estimates for RC_t:
rho0_hat      = 17.826900
rho1_hat      = 0.624909
sigma_rho_hat = 4.605933

Occupied states = 35 out of 35

================ NFXP results ================
success                = True
message                = Optimization terminated successfully.
loglike                = -34101.683117
theta1_hat             = 0.407192
theta2_hat             = 0.154317
runtime_sec            = 0.2543
last_inner_iterations  = 1

============= Hotz-Miller results =============
success                = False
message                = Desired error not necessarily achieved due to precision loss.
loglike                = -34103.991368
theta1_hat             = 0.404597
theta2_hat             = 0.153985
runtime_sec            = 0.0117
HM value solve         = direct linear solve

================ Summary table ================
         method  theta1_hat  theta2_hat  rho0_hat  rho1_hat  sigma_rho_hat       loglike  runtime_sec  

In [ ]:
# ============================================================
#  Part (c): Forward simulation estimators
#  REPLACE your old part-(c) block with this entire block.
#  This version fixes two issues:
#  1. it vectorizes across all states and simulation replications
#     so there is no nested Python loop over states and replications;
#  2. it stabilizes (c.2) by using more simulation draws, antithetic
#     Type-I EV shocks, and by starting from the (c.1) estimate.
# ============================================================

# ------------------------------------------------------------
# 0. Simulation tuning
# ------------------------------------------------------------
# Keep (c.1) at the homework suggestion level.
Nsim_c1 = 100

# Use more draws for (c.2), because direct shock simulation has
# much higher Monte Carlo variance than (c.1).
# Choose an even number so antithetic pairing is easy.
Nsim_c2 = 400

# A horizon of 120 is already enough because beta^120 is tiny.
Tsim = 120

# Fixed seed for common random numbers.
# This makes the simulated criterion deterministic across optimizer
# iterations, which is very important for numerical stability.
sim_seed = 6622026
rng_sim = np.random.default_rng(sim_seed)

# ------------------------------------------------------------
# 1. State helpers under your existing ordering:
#    state = x_index * K + g_index
#    with x_index in {0,...,6}, g_index in {0,...,K-1}
# ------------------------------------------------------------
x_idx_of_state = np.repeat(np.arange(7, dtype=np.int16), K)
g_idx_of_state = np.tile(np.arange(K, dtype=np.int16), 7)

# First-stage CCP from your HM-style step-2 smoothing.
p1_policy = ccp_smooth.copy()
p0_policy = 1.0 - p1_policy

# Log-odds of the first-stage CCP, needed for (c.2).
logit_policy = np.log(p1_policy) - np.log(p0_policy)

# RC-grid transition CDF for inverse-CDF simulation.
Q_rc_cdf = np.cumsum(Q_rc, axis=1)
Q_rc_cdf[:, -1] = 1.0

# ------------------------------------------------------------
# 2. Initial state matrices for vectorized simulation
#    Row s corresponds to initial state s.
#    Column r corresponds to simulation replication r.
# ------------------------------------------------------------
x0_c1 = np.repeat(x_idx_of_state[:, None], Nsim_c1, axis=1)
g0_c1 = np.repeat(g_idx_of_state[:, None], Nsim_c1, axis=1)

x0_c2 = np.repeat(x_idx_of_state[:, None], Nsim_c2, axis=1)
g0_c2 = np.repeat(g_idx_of_state[:, None], Nsim_c2, axis=1)

# ------------------------------------------------------------
# 3. Common random numbers
# ------------------------------------------------------------

# ---------- (c.1): action uniforms and RC uniforms ----------
u_action_c1 = rng_sim.random((S, Nsim_c1, Tsim + 1), dtype=np.float64)

# Use antithetic uniforms for the RC transitions as well.
half_c1 = Nsim_c1 // 2
u_rc_half_c1 = rng_sim.random((S, half_c1, Tsim), dtype=np.float64)
u_rc_c1 = np.concatenate([u_rc_half_c1, 1.0 - u_rc_half_c1], axis=1)

# If Nsim_c1 is odd, append one extra column.
if u_rc_c1.shape[1] < Nsim_c1:
    extra = rng_sim.random((S, 1, Tsim), dtype=np.float64)
    u_rc_c1 = np.concatenate([u_rc_c1, extra], axis=1)

# ---------- (c.2): antithetic Gumbel shocks and RC uniforms ----------
half_c2 = Nsim_c2 // 2

# Build antithetic uniforms, then transform to Type-I EV shocks.
# Standard Type-I EV shock: eps = -log(-log(U)).
u0_half = rng_sim.random((S, half_c2, Tsim + 1), dtype=np.float64)
u1_half = rng_sim.random((S, half_c2, Tsim + 1), dtype=np.float64)

u0_full = np.concatenate([u0_half, 1.0 - u0_half], axis=1)
u1_full = np.concatenate([u1_half, 1.0 - u1_half], axis=1)

# Guard against exact 0 or 1 in the inverse transform.
u0_full = np.clip(u0_full, 1e-12, 1.0 - 1e-12)
u1_full = np.clip(u1_full, 1e-12, 1.0 - 1e-12)

eps0_c2 = -np.log(-np.log(u0_full))
eps1_c2 = -np.log(-np.log(u1_full))

# Antithetic RC draws for (c.2)
u_rc_half_c2 = rng_sim.random((S, half_c2, Tsim), dtype=np.float64)
u_rc_c2 = np.concatenate([u_rc_half_c2, 1.0 - u_rc_half_c2], axis=1)

if u_rc_c2.shape[1] < Nsim_c2:
    extra = rng_sim.random((S, 1, Tsim), dtype=np.float64)
    u_rc_c2 = np.concatenate([u_rc_c2, extra], axis=1)

# ------------------------------------------------------------
# 4. Vectorized RC-grid transition sampler
#    Input:
#      g_idx_mat : shape (S, R)
#      u_mat     : shape (S, R)
#    Output:
#      next g    : shape (S, R)
# ------------------------------------------------------------
def draw_next_rc_grid_mat(g_idx_mat, u_mat):
    # Q_rc_cdf[g_idx_mat] has shape (S, R, K)
    cdf_rows = Q_rc_cdf[g_idx_mat]

    # Count how many cumulative probabilities are below the draw.
    # That count is exactly the sampled grid index.
    g_next = np.sum(u_mat[:, :, None] > cdf_rows, axis=2)

    return np.clip(g_next, 0, K - 1).astype(np.int16)

# ------------------------------------------------------------
# 5. Fast simulator for (c.1)
#    Only loop over time; states and replications are vectorized.
# ------------------------------------------------------------
def simulate_value_c1_fast(theta1, theta2):
    # Current states for all initial states and all replications.
    x_idx = x0_c1.copy()
    g_idx = g0_c1.copy()

    # Running return matrix: row = initial state, col = replication.
    G = np.zeros((S, Nsim_c1), dtype=np.float64)

    # Discount factor beta^tau
    discount = 1.0

    for tau in range(Tsim + 1):
        # Current state id for every (initial state, replication)
        state_now = x_idx * K + g_idx

        # First-stage CCP at the current state
        p1_now = p1_policy[state_now]

        # Draw action from the estimated CCP
        a_now = (u_action_c1[:, :, tau] < p1_now)

        # Chosen-action probability under the first-stage policy
        p_chosen = np.where(a_now, p1_now, 1.0 - p1_now)

        # Deterministic utility \bar{u}(s,a;theta)
        ubar_now = np.where(
            a_now,
            theta2 * u1_base_vec[state_now],
            theta1 * u0_base_vec[state_now]
        )

        # Eq. (5): \bar{u} + gamma - log P(a|s)
        G += discount * (ubar_now + gamma_euler - np.log(p_chosen))

        if tau < Tsim:
            # Mileage transition
            x_idx = np.where(a_now, 0, x_next_keep[x_idx]).astype(np.int16)

            # RC-grid transition
            g_idx = draw_next_rc_grid_mat(g_idx, u_rc_c1[:, :, tau])

            discount *= beta

    # Average across replications to get V_hat(s;theta)
    return G.mean(axis=1)

# ------------------------------------------------------------
# 6. Fast simulator for (c.2)
#    Again only loop over time.
# ------------------------------------------------------------
def simulate_value_c2_fast(theta1, theta2):
    # Current states for all initial states and all replications.
    x_idx = x0_c2.copy()
    g_idx = g0_c2.copy()

    # Running return matrix
    G = np.zeros((S, Nsim_c2), dtype=np.float64)

    # Discount factor beta^tau
    discount = 1.0

    for tau in range(Tsim + 1):
        # Current state id
        state_now = x_idx * K + g_idx

        # First-stage log-odds at the current state
        logit_now = logit_policy[state_now]

        # Direct Type-I EV shocks
        e0_now = eps0_c2[:, :, tau]
        e1_now = eps1_c2[:, :, tau]

        # Eq. (6): choose action by the threshold rule
        a_now = (logit_now + e1_now - e0_now > 0.0)

        # Deterministic utility \bar{u}(s,a;theta)
        ubar_now = np.where(
            a_now,
            theta2 * u1_base_vec[state_now],
            theta1 * u0_base_vec[state_now]
        )

        # Selected shock epsilon_a
        eps_selected = np.where(a_now, e1_now, e0_now)

        # Eq. (7): \bar{u}(s,a;theta) + epsilon_a
        G += discount * (ubar_now + eps_selected)

        if tau < Tsim:
            # Mileage transition
            x_idx = np.where(a_now, 0, x_next_keep[x_idx]).astype(np.int16)

            # RC-grid transition
            g_idx = draw_next_rc_grid_mat(g_idx, u_rc_c2[:, :, tau])

            discount *= beta

    # Average across replications to get V_hat(s;theta)
    return G.mean(axis=1)

# ------------------------------------------------------------
# 7. Objective functions
#    Same state-level grouped log-likelihood as in your NFXP/HM code.
# ------------------------------------------------------------
def sim_c1_objective_eta(eta):
    theta = np.exp(eta)
    theta1 = float(theta[0])
    theta2 = float(theta[1])

    # Simulated value under the first-stage policy
    V_sim = simulate_value_c1_fast(theta1, theta2)

    # Recover EV_a(s;theta) through F0 and F1 and then form the
    # structural CCP implied by theta and V_sim.
    p1_model, _, _, _, _ = choice_prob_from_V_flat(theta1, theta2, V_sim)

    ll = np.sum(N1_vec * np.log(p1_model) + N0_vec * np.log1p(-p1_model))
    return -ll

def sim_c2_objective_eta(eta):
    theta = np.exp(eta)
    theta1 = float(theta[0])
    theta2 = float(theta[1])

    V_sim = simulate_value_c2_fast(theta1, theta2)
    p1_model, _, _, _, _ = choice_prob_from_V_flat(theta1, theta2, V_sim)

    ll = np.sum(N1_vec * np.log(p1_model) + N0_vec * np.log1p(-p1_model))
    return -ll

# ------------------------------------------------------------
# 8. Estimate (c.1)
# ------------------------------------------------------------
# Start from HM for (c.1)
eta_start_c1 = res_hm.x.copy()

# Bounds on eta = log(theta)
eta_bounds = [(-6.0, 2.0), (-6.0, 2.0)]

t0 = time.perf_counter()

res_c1 = minimize(
    sim_c1_objective_eta,
    eta_start_c1,
    method="L-BFGS-B",
    bounds=eta_bounds,
    options={
        "maxiter": 100,
        "ftol": 1e-10,
        "eps": 1e-4,   # finite-difference step for a smooth deterministic objective
        "disp": False,
    },
)

time_c1 = time.perf_counter() - t0

theta_c1 = np.exp(res_c1.x)
V_c1 = simulate_value_c1_fast(float(theta_c1[0]), float(theta_c1[1]))
p1_c1, _, _, _, _ = choice_prob_from_V_flat(float(theta_c1[0]), float(theta_c1[1]), V_c1)
ll_c1 = np.sum(N1_vec * np.log(p1_c1) + N0_vec * np.log1p(-p1_c1))

# ------------------------------------------------------------
# 9. Estimate (c.2)
# ------------------------------------------------------------
# IMPORTANT:
# Start (c.2) from the (c.1) estimate, not from HM.
# Since (c.2) is noisier, this materially improves convergence.
eta_start_c2 = res_c1.x.copy()

t0 = time.perf_counter()

res_c2 = minimize(
    sim_c2_objective_eta,
    eta_start_c2,
    method="L-BFGS-B",
    bounds=eta_bounds,
    options={
        "maxiter": 80,
        "ftol": 1e-10,
        "eps": 5e-4,   # slightly larger step helps with the noisier (c.2) objective
        "disp": False,
    },
)

time_c2 = time.perf_counter() - t0

theta_c2 = np.exp(res_c2.x)
V_c2 = simulate_value_c2_fast(float(theta_c2[0]), float(theta_c2[1]))
p1_c2, _, _, _, _ = choice_prob_from_V_flat(float(theta_c2[0]), float(theta_c2[1]), V_c2)
ll_c2 = np.sum(N1_vec * np.log(p1_c2) + N0_vec * np.log1p(-p1_c2))

# ------------------------------------------------------------
# 10. Reports
# ------------------------------------------------------------
print("\n============= Forward simulation (c.1) results =============")
print(f"success                = {res_c1.success}")
print(f"message                = {res_c1.message}")
print(f"loglike                = {ll_c1:.6f}")
print(f"theta1_hat             = {theta_c1[0]:.6f}")
print(f"theta2_hat             = {theta_c1[1]:.6f}")
print(f"runtime_sec            = {time_c1:.4f}")
print(f"Nsim                   = {Nsim_c1}")
print(f"Tsim                   = {Tsim}")

print("\n============= Forward simulation (c.2) results =============")
print(f"success                = {res_c2.success}")
print(f"message                = {res_c2.message}")
print(f"loglike                = {ll_c2:.6f}")
print(f"theta1_hat             = {theta_c2[0]:.6f}")
print(f"theta2_hat             = {theta_c2[1]:.6f}")
print(f"runtime_sec            = {time_c2:.4f}")
print(f"Nsim                   = {Nsim_c2}")
print(f"Tsim                   = {Tsim}")

# Extend your summary table
summary_part_c = pd.DataFrame({
    "method": ["Forward simulation (c.1)", "Forward simulation (c.2)"],
    "theta1_hat": [theta_c1[0], theta_c2[0]],
    "theta2_hat": [theta_c1[1], theta_c2[1]],
    "rho0_hat": [rho0_hat, rho0_hat],
    "rho1_hat": [rho1_hat, rho1_hat],
    "sigma_rho_hat": [sigma_rho_hat, sigma_rho_hat],
    "loglike": [ll_c1, ll_c2],
    "runtime_sec": [time_c1, time_c2],
    "state_space_S": [S, S],
})

summary_all = pd.concat([summary, summary_part_c], ignore_index=True)

print("\n================ Updated summary table ================")
print(summary_all.to_string(index=False))

print("\nFirst 10 occupied-state model CCPs from forward simulation (c.1):")
print(p1_c1[occupied][:10])

print("\nFirst 10 occupied-state model CCPs from forward simulation (c.2):")
print(p1_c2[occupied][:10])

/tmp/ipykernel_8321/3594278512.py:267: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res_c1 = minimize(
/tmp/ipykernel_8321/3594278512.py:297: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res_c2 = minimize(



============= Forward simulation (c.1) results =============
success                = True
message                = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
loglike                = -34131.674999
theta1_hat             = 0.402939
theta2_hat             = 0.152693
runtime_sec            = 10.0160
Nsim                   = 100
Tsim                   = 120

============= Forward simulation (c.2) results =============
success                = True
message                = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
loglike                = -34153.566228
theta1_hat             = 0.415577
theta2_hat             = 0.156439
runtime_sec            = 24.8342
Nsim                   = 400
Tsim                   = 120

================ Updated summary table ================
                  method  theta1_hat  theta2_hat  rho0_hat  rho1_hat  sigma_rho_hat       loglike  runtime_sec  state_space_S
                    NFXP    0.407192    0.154317   17.8269  0.624909       4.6059

## Apendix A

Deriving the distribution of the maximum:

Denote $M = \max_a (\delta_a + \varepsilon_a)$ where in the case of our specific question $\delta_a = \bar{u}(s, a) + \beta EV_a(s)$. Then we have:

$$
\Pr(M \leq m) = \Pr(\delta_a + \varepsilon_a \leq m \ \forall a)
$$

Equivalently,

$$
\Pr(M \leq m) = \prod_a \Pr(\varepsilon_a \leq m - \delta_a)
$$

using independence condition. Then for type-1 extreme distribution with cdf, we have:

$$
F(\varepsilon) = \exp(-e^{-\varepsilon})
$$

Plugging back, we have:

$$
\Pr(M \leq m) = \exp(-e^{-m}  \sum_a e^{\delta_a}) = \exp(-\exp[-(m - \log \sum_a \delta^a)])
$$

So the maximum $M$ is itself Gumbel-distributed with location $\mu = \log \sum_a e^{\delta_a}$. Therefore, we can write:

$$
\mathbb{E}[M] = \log \sum_a e^{\delta_a} + \gamma
$$

Substitute back with $\delta_a$ then we can derive the result.

# Appendix B

